In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 156 (delta 58), reused 132 (delta 36), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 19.98 MiB | 24.65 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import time
import numpy as np
import centpy

In [8]:
class Euler2d(centpy.Equation2d):

    def pressure(self, u):
        return (self.gamma - 1.0) * (
            u[:, :, 3] - 0.5 * (u[:, :, 1] ** 2 + u[:, :, 2] ** 2) / u[:, :, 0]
        )

    def euler_data(self):
        gamma = self.gamma
        p_one, p_two, p_three, p_four = 1.5, 0.3, 0.029, 0.3

        upper_right, upper_left, lower_right, lower_left = np.ones((4, 4))

        upper_right[0] = 1.5
        upper_right[1] = 0.0
        upper_right[2] = 0.0
        upper_right[3] = (p_one / (gamma - 1.0) + 0.5 * (upper_right[1]**2 + upper_right[2]**2) / upper_right[0])

        upper_left[0] = 0.5323
        upper_left[1] = 1.206 * upper_left[0]
        upper_left[2] = 0.0
        upper_left[3] = (p_two / (gamma - 1.0) + 0.5 * (upper_left[1]**2 + upper_left[2]**2) / upper_left[0])

        lower_right[0] = 0.5323
        lower_right[1] = 0.0
        lower_right[2] = 1.206 * lower_right[0]
        lower_right[3] = (p_four / (gamma - 1.0) + 0.5 * (lower_right[1]**2 + lower_right[2]**2) / lower_right[0])

        lower_left[0] = 0.138
        lower_left[1] = 1.206 * lower_left[0]
        lower_left[2] = 1.206 * lower_left[0]
        lower_left[3] = (p_three / (gamma - 1.0) + 0.5 * (lower_left[1]**2 + lower_left[2]**2) / lower_left[0])

        return upper_right, upper_left, lower_right, lower_left

    def initial_data(self):
        u = np.empty((self.J + 4, self.K + 4, 4))
        midJ = int(self.J / 2) + 2
        midK = int(self.K / 2) + 2

        one_matrix = np.ones(u[midJ:, midK:].shape)
        upper_right, upper_left, lower_right, lower_left = self.euler_data()

        u[midJ:, midK:] = upper_right * one_matrix
        u[:midJ, midK:] = upper_left * one_matrix
        u[midJ:, :midK] = lower_right * one_matrix
        u[:midJ, :midK] = lower_left * one_matrix
        return u

    def boundary_conditions(self, u):
        upper_right, upper_left, lower_right, lower_left = self.euler_data()

        if self.odd:
            j = slice(1, -2)
            u[j, 0], u[j, -2], u[j, -1] = u[j, 1], u[j, -3], u[j, -3]
            u[0, j], u[-2, j], u[-1, j] = u[1, j], u[-3, j], u[-3, j]

            u[-2, -2], u[-1, -2], u[-2, -1], u[-1, -1] = upper_right, upper_right, upper_right, upper_right
            u[0, -2], u[0, -1] = upper_left, upper_left
            u[0, 0], u[0, 1], u[1, 0], u[1, 1] = lower_left, lower_left, lower_left, lower_left
            u[-2, 0], u[-1, 0], u[-2, 1], u[-1, 1] = lower_right, lower_right, lower_right, lower_right
        else:
            j = slice(2, -1)
            u[j, 0], u[j, 1], u[j, -1] = u[j, 2], u[j, 2], u[j, -2]
            u[0, j], u[1, j], u[-1, j] = u[2, j], u[2, j], u[-2, j]

            u[-1, -2], u[-1, -1] = upper_right, upper_right
            u[0, -2], u[0, -1], u[1, -2], u[1, -1] = upper_left, upper_left, upper_left, upper_left
            u[0, 0], u[0, 1], u[1, 0], u[1, 1] = lower_left, lower_left, lower_left, lower_left
            u[-1, 0], u[-1, 1] = lower_right, lower_right

    def flux_x(self, u):
        f = np.empty_like(u)
        p = self.pressure(u)
        f[:, :, 0] = u[:, :, 1]
        f[:, :, 1] = u[:, :, 1] ** 2 / u[:, :, 0] + p
        f[:, :, 2] = u[:, :, 1] * u[:, :, 2] / u[:, :, 0]
        f[:, :, 3] = (u[:, :, 3] + p) * u[:, :, 1] / u[:, :, 0]
        return f

    def flux_y(self, u):
        g = np.empty_like(u)
        p = self.pressure(u)
        g[:, :, 0] = u[:, :, 2]
        g[:, :, 1] = u[:, :, 1] * u[:, :, 2] / u[:, :, 0]
        g[:, :, 2] = u[:, :, 2] ** 2 / u[:, :, 0] + p
        g[:, :, 3] = (u[:, :, 3] + p) * u[:, :, 2] / u[:, :, 0]
        return g

    def spectral_radius_x(self, u):
        j0 = slice(2, -2) # Исправлено отсутствие centpy.helpers
        rho = u[j0, j0, 0]
        vx = u[j0, j0, 1] / rho
        vy = u[j0, j0, 2] / rho
        p = (self.gamma - 1.0) * (u[j0, j0, 3] - 0.5 * rho * (vx ** 2 + vy ** 2))
        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vx) + c

    def spectral_radius_y(self, u):
        j0 = slice(2, -2) # Исправлено отсутствие centpy.helpers
        rho = u[j0, j0, 0]
        vx = u[j0, j0, 1] / rho
        vy = u[j0, j0, 2] / rho
        p = (self.gamma - 1.0) * (u[j0, j0, 3] - 0.5 * rho * (vx ** 2 + vy ** 2))
        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vy) + c


def run_cpu_benchmark():
    # Параметры из оригинального примера Euler_2d
    pars = centpy.Pars2d(
        x_init=0., x_final=1.,
        y_init=0., y_final=1.,
        J=200, K=200,
        t_final=0.4,
        dt_out=0.005,
        cfl=0.475,
        scheme="fd2"
    )
    pars.gamma = 1.4

    eqn = Euler2d(pars)
    solver = centpy.Solver2d(eqn)

    print("--- Запуск CPU Бенчмарка (Euler 2D, сетка 200x200) ---")

    t0 = time.time()
    solver.solve()
    t1 = time.time()

    print(f"\n[CPU centpy] Полное время выполнения Эйлера: {t1 - t0:.4f} секунд")

if __name__ == "__main__":
    run_cpu_benchmark()

--- Запуск CPU Бенчмарка (Euler 2D, сетка 200x200) ---

[CPU centpy] Полное время выполнения Эйлера: 31.2573 секунд
